In [3]:
# input
import pandas as pd
import argparse
import numpy as np

In [9]:
def get_peaks(data,bin_size_kb=2, comp_flanksize_kb=50, fold_thresh=5):
    scaff_l = list(set(data.Scaffold))
    peak = []
    nopeak = []
    
    for scaff in scaff_l:
        onekbfile = data.loc[data.Scaffold==scaff].reset_index()
        
        for i, bin in onekbfile.iterrows():
            #print(i)
            #print(scaff)
            #print(bin.Start_bp)
            #print(onekbfile)
            #print(onekbfile.iloc[i:i+bin_size_kb-1])
            peak_mean_rho = onekbfile.iloc[i:i+bin_size_kb-1].Rho_kb.mean()
            #print(peak_mean_rho)

            if i == onekbfile.index.max():
                peak_end = onekbfile.iloc[i,].End_bp
            else:
                try:
                    peak_end = onekbfile.iloc[i+1,].End_bp
                except IndexError:
                    #print(onekbfile.index.max())
                    #print(i)
                    #print(bin)
                    peak_end = onekbfile.iloc[i,].End_bp
                    raise IndexError
                    
                    
            flank_start = i-50
            if flank_start<0: # make sure we dont overshoot the scaffold boundary
                flank_start=0
    
            flank_stop = i+bin_size_kb-1+50
            if flank_stop>onekbfile.index.max(): # make sure we dont overshoot the scaffold boundary
                flank_stop=onekbfile.index.max()

            flank_mean_rho = onekbfile.iloc[flank_start:flank_stop].Rho_kb.mean()
            #print(scaff)
            #print(bin.Start_bp)
            #print(peak_end)
            #print(peak_mean_rho)
            #print(flank_mean_rho)
            #print(peak_mean_rho/flank_mean_rho)
            fold_diff = (peak_mean_rho/flank_mean_rho)
            if fold_diff>fold_thresh:
                #print("PEAK")
                peak.append([scaff, bin.Start_bp, peak_end, True, fold_diff, peak_mean_rho, flank_mean_rho])
            else:
                nopeak.append([scaff, bin.Start_bp, peak_end, False, fold_diff, peak_mean_rho, flank_mean_rho])
        #    else: 
        #        peak.append('False')
    return peak, nopeak
        
    

In [5]:
data = pd.read_csv("../data/concat_Mbel_excl5scaff_rmind_hardfilt_exhet_biall_dp_qfilt_mac2_maxmiss06_phimp_LDhat_bpen1_statres_100N_filtMQ70depth2stdev.txt_w1kb", sep='\t')

In [6]:
p, pn = get_peaks(data=data)

In [18]:
peaks = pd.DataFrame(p)
nopeaks = pd.DataFrame(pn)

In [20]:
peaks.columns = ["scaffold", "bin_Start_bp", "bin_Stop_bp", "peak", "fold_diff", "peak_mean_rho", "flank_mean_rho"]
nopeaks.columns = ["scaffold", "bin_Start_bp", "bin_Stop_bp", "no_peak", "fold_diff", "peak_mean_rho", "flank_mean_rho"]

In [21]:
peaks.bin_Start_bp = peaks.bin_Start_bp.astype(int)#.astype(str)
peaks.bin_Stop_bp = peaks.bin_Stop_bp.astype(int)#.astype(str)
nopeaks.bin_Start_bp = nopeaks.bin_Start_bp.astype(int)#.astype(str)
nopeaks.bin_Stop_bp = nopeaks.bin_Stop_bp.astype(int)#.astype(str)

In [22]:
peaks.to_csv('../data/20231204_Mbel_5fold_peaks.bed', sep='\t', header=None, index=None)


In [2]:
peaks.shape

NameError: name 'peaks' is not defined

In [1]:
nopeaks.shape

NameError: name 'nopeaks' is not defined

In [25]:
take_idx = np.random.choice( a= range(nopeaks.index.max()), size=peaks.shape[0], replace=False )
nopeaks_ss = nopeaks.iloc[take_idx]

In [26]:
nopeaks_ss[["scaffold", "bin_Start_bp",	"bin_Stop_bp"]].to_csv('../data/20231205_Mbel_5fold_nopeaks.bed', sep='\t', header=None, index=None)

In [30]:
peaks.flank_mean_rho.mean()

1.5595829390120386

In [31]:
peaks_ss = peaks.loc[peaks.flank_mean_rho > 1.5595829390120386].sort_values(by='fold_diff', ascending=False)

In [32]:
peaks_ss

,scaffold,bin_Start_bp,bin_Stop_bp,peak,fold_diff,peak_mean_rho,flank_mean_rho
6552,scaffold56,5563000,5565000,True,62.563174,118.11800,1.887980
9177,scaffold108,864000,866000,True,48.347053,77.62150,1.605506
25814,scaffold2,12809000,12811000,True,42.080825,66.54080,1.581262
15107,scaffold12,24858000,24860000,True,39.475393,66.37680,1.681473
30031,scaffold7,21538000,21540000,True,38.881003,100.17700,2.576502
...,...,...,...,...,...,...,...
10971,scaffold1,39371000,39373000,True,5.000767,41.88470,8.375655
25989,scaffold2,18631000,18633000,True,5.000330,9.81745,1.963360
19474,scaffold37,9458000,9460000,True,5.000274,10.59730,2.119344
5360,scaffold20,3803000,3805000,True,5.000179,15.60920,3.121728


In [35]:
take_idx = np.random.choice( a= range(nopeaks.index.max()), size=peaks_ss.shape[0], replace=False )
nopeaks_ss = nopeaks.iloc[take_idx]

In [37]:
peaks_ss.to_csv('../data/20231204_Mbel_5fold_peaks_13kss.bed', sep='\t', header=None, index=None)
nopeaks_ss[["scaffold", "bin_Start_bp",	"bin_Stop_bp"]].to_csv('../data/20231205_Mbel_5fold_nopeaks_13kss.bed', sep='\t', header=None, index=None)

In [38]:
%%bash

db=../data/New_Mbel.fa
bed=../data/20231204_Mbel_5fold_peaks_13kss.bed
out=../data/20231204_Mbel_hs_5fold_13kss.fa

bedtools getfasta -fi $db -bed $bed -fo $out


db=../data/New_Mbel.fa
bed=../data/20231205_Mbel_5fold_nopeaks_13kss.bed
out=../data/20231204_Mbel_hs_5fold_background_13kss.fa

bedtools getfasta -fi $db -bed $bed -fo $out


Feature (scaffold22:21018000-21019000) beyond the length of scaffold22 size (21018719 bp).  Skipping.
Feature (scaffold22:21017000-21019000) beyond the length of scaffold22 size (21018719 bp).  Skipping.


In [28]:
seqs = pd.read_csv("../data/streme_out_v2/sequences.tsv", sep='\t')
seqs = seqs.dropna(subset='motif_ALT_ID')
seqs['idnum'] = [ int(i.split('-')[1]) for i in seqs.motif_ALT_ID]
seqs.loc[seqs.idnum<8][['motif_ID', 'motif_P-value']].drop_duplicates()

,motif_ID,motif_P-value
0,1-GCATTATCTATAGACA,0.000015
163,2-GTTAWGCCCTTATGAC,0.000240
459,3-AGCTCAGTAACGTCG,0.000330
746,4-CGCACTGGTTAGCC,0.000340
911,5-CGTCCTCTT,0.000370
3146,6-ACGTCATACCCTGAAG,0.000380
3523,7-GAGTTGTAATCCCA,0.000600
